# GALVAT Trial 8: append Roboflow sources to the Roboflow-only checkpoint

This notebook starts from the Roboflow-only Trial 8 checkpoint:

`/content/drive/MyDrive/galvat_trial8_dataset_checkpoints/galvat_trial8_roboflow_only_from_trial7_2026-06-26_02-18-37.zip`

It then lets you paste new Roboflow download URLs/scripts for missing or underrepresented classes, imports only the target class labels, appends the images without overwriting existing files, audits the dataset, and saves a new checkpoint zip to Drive.

This notebook keeps the current 44-class ontology from the checkpoint. Bathroom-sign splitting is intentionally not done here.


In [ ]:
# Cell 1: Setup
!pip -q install pyyaml pandas pillow tqdm matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import datetime
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from PIL import Image, ImageOps
from tqdm.auto import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


In [ ]:
# Cell 2: Paths and goals

SOURCE_ZIP = Path('/content/drive/MyDrive/galvat_trial8_dataset_checkpoints/galvat_trial8_roboflow_only_from_trial7_2026-06-26_02-18-37.zip')

WORK_ROOT = Path('/content/galvat_trial8_append_roboflow_sources')
RESTORE_ROOT = WORK_ROOT / 'restore'
WORK_DS = WORK_ROOT / 'dataset'
WORK_META = WORK_ROOT / 'metadata'
ROBOSRC_ROOT = Path('/content/roboflow_sources_trial8_append')
DRIVE_OUT = Path('/content/drive/MyDrive/galvat_trial8_dataset_checkpoints')

for p in [RESTORE_ROOT, WORK_META, ROBOSRC_ROOT, DRIVE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# Image-level goals. The notebook computes current counts after unzip and imports only what is still needed.
TARGET_IMAGES = {
    'urinal': 1500, 'paper_towel_dispenser': 1500, 'light_switch': 1500, 'railing': 1500,
    'hearing_aid': 1500, 'step_curb': 1500, 'exit_sign': 1500, 'bathroom_sign': 1500,
    'door': 1500, 'door_frame': 1500, 'electrical_outlet': 1500,
    'tupperware': 1000, 'jug': 1000, 'canned_good': 1000, 'fork': 1000, 'spoon': 1000,
    'knife': 1000, 'drinking_glass': 1000, 'wine_glass': 1000, 'trash_can': 1000,
    'coffee_mug': 1000, 'salt_pepper_shaker': 1000, 'measuring_cup': 1000, 'pan': 1000,
    'cellphone': 1000, 'purse': 1000, 'toilet': 1000, 'toothbrush': 1000, 'tv_remote': 1000,
    'writing_device': 1000, 'eyeglasses': 1000, 'thermostat': 1000, 'razor': 1000,
    'window': 1000, 'chair': 1000, 'table': 1000, 'soap_dispenser': 1000, 'wallet': 1000,
    'toothpaste': 1000, 'medicine_bottle': 1000, 'lemon': 1000, 'deodorant': 1000,
    'keys': 1000, 'pot': 1000, 'cane': 1000,
}

assert SOURCE_ZIP.exists(), f'Missing checkpoint zip: {SOURCE_ZIP}'
print('Source checkpoint:', SOURCE_ZIP)
print('Work root:', WORK_ROOT)


In [ ]:
# Cell 3: Unzip checkpoint to /content

if WORK_DS.exists() and any(WORK_DS.glob('*')):
    print('Working dataset already exists:', WORK_DS)
    print('If you need a clean rerun, delete WORK_ROOT manually and rerun.')
else:
    print('Unzipping:', SOURCE_ZIP)
    with zipfile.ZipFile(SOURCE_ZIP, 'r') as z:
        z.extractall(RESTORE_ROOT)

    data_yaml_candidates = sorted(RESTORE_ROOT.rglob('data.yaml'))
    assert data_yaml_candidates, f'No data.yaml found under {RESTORE_ROOT}'
    BASE_DATA_YAML = data_yaml_candidates[0]
    BASE_DS = BASE_DATA_YAML.parent
    shutil.copytree(BASE_DS, WORK_DS, dirs_exist_ok=True)
    print('Base dataset:', BASE_DS)
    print('Copied working dataset:', WORK_DS)

DATA_YAML = WORK_DS / 'data.yaml'
assert DATA_YAML.exists(), f'Missing data.yaml: {DATA_YAML}'


In [ ]:
# Cell 4: Load class names and audit helpers

def load_yolo_names(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names')
    if isinstance(names, dict):
        names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
    return data, list(names)

data_yaml, NAMES = load_yolo_names(DATA_YAML)
data_yaml['path'] = str(WORK_DS)
data_yaml['train'] = 'train/images'
data_yaml['val'] = 'valid/images'
data_yaml['test'] = 'test/images'
data_yaml['names'] = {i: name for i, name in enumerate(NAMES)}
data_yaml['nc'] = len(NAMES)
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

name_to_id = {name: i for i, name in enumerate(NAMES)}

def label_file_for_image(img_path):
    return Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')

def audit_dataset():
    split_rows = []
    box_counts = Counter()
    image_counts = Counter()
    empty_counts = Counter()
    bad_rows = []
    for split in ['train', 'valid', 'test']:
        img_dir = WORK_DS / split / 'images'
        img_files = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else []
        for img_path in img_files:
            lbl_path = label_file_for_image(img_path)
            lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
            seen = set()
            valid = 0
            for line_no, line in enumerate(lines, start=1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_column_count', 'raw': line})
                    continue
                try:
                    cls_id = int(float(parts[0])); x, y, w, h = [float(v) for v in parts[1:]]
                except Exception:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'parse_error', 'raw': line})
                    continue
                if cls_id < 0 or cls_id >= len(NAMES):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_class_id', 'raw': line})
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_coords', 'raw': line})
                    continue
                cls = NAMES[cls_id]
                box_counts[(split, cls)] += 1
                seen.add(cls)
                valid += 1
            if valid == 0:
                empty_counts[split] += 1
            for cls in seen:
                image_counts[(split, cls)] += 1
        split_rows.append({'split': split, 'images': len(img_files), 'empty_label_images': empty_counts[split]})

    class_rows = []
    for cls in NAMES:
        total_images = sum(image_counts[(s, cls)] for s in ['train', 'valid', 'test'])
        total_boxes = sum(box_counts[(s, cls)] for s in ['train', 'valid', 'test'])
        goal = TARGET_IMAGES.get(cls, 1000)
        class_rows.append({
            'class': cls,
            'train_images': image_counts[('train', cls)],
            'valid_images': image_counts[('valid', cls)],
            'test_images': image_counts[('test', cls)],
            'total_images': total_images,
            'train_boxes': box_counts[('train', cls)],
            'valid_boxes': box_counts[('valid', cls)],
            'test_boxes': box_counts[('test', cls)],
            'total_boxes': total_boxes,
            'goal_images': goal,
            'still_need': max(0, goal - total_images),
        })
    return pd.DataFrame(split_rows), pd.DataFrame(class_rows).sort_values(['still_need', 'total_images'], ascending=[False, True]), pd.DataFrame(bad_rows)

split_summary, class_summary, bad_labels = audit_dataset()
display(split_summary)
display(class_summary)
print('Class count:', len(NAMES))
print('Bad label rows:', len(bad_labels))


In [ ]:
# Cell 5: Add Roboflow sources here
#
# Paste either the raw URL or the whole curl command into url_or_script.
# Any entry with a non-empty url_or_script is enabled automatically.
# Add 'skip': True when you want to keep a pasted source disabled temporarily.
# You can add multiple sources for the same target_class; imports append and do not overwrite existing images.
# Give additional sources unique source_name values, such as chair_2 or chair_office.
#
# Optional mapping override for unusual data.yaml files:
# 'source_class_ids': [0]
# Use this only after Cell 6 shows which source id belongs to the desired class.
#
# Multi-class safety:
# 'reject_mixed_class_images': True is the default. It prevents importing an
# image when it also contains source labels that are not mapped to target_class.
# This avoids creating unlabeled positives in the combined dataset.

ROBOFLOW_SOURCES = [
    {'enabled': False, 'source_name': 'urinal', 'target_class': 'urinal', 'url_or_script': '', 'aliases': ['urinal'], 'target_total_images': TARGET_IMAGES['urinal']},
    {'enabled': False, 'source_name': 'paper_towel_dispenser', 'target_class': 'paper_towel_dispenser', 'url_or_script': '', 'aliases': ['paper_towel_dispenser', 'paper towel dispenser'], 'target_total_images': TARGET_IMAGES['paper_towel_dispenser']},
    {'enabled': False, 'source_name': 'light_switch', 'target_class': 'light_switch', 'url_or_script': '', 'aliases': ['light_switch', 'light switch'], 'target_total_images': TARGET_IMAGES['light_switch']},
    {'enabled': False, 'source_name': 'railing', 'target_class': 'railing', 'url_or_script': '', 'aliases': ['railing', 'handrail', 'rail'], 'target_total_images': TARGET_IMAGES['railing']},
    {'enabled': False, 'source_name': 'hearing_aid', 'target_class': 'hearing_aid', 'url_or_script': '', 'aliases': ['hearing_aid', 'hearing aid'], 'target_total_images': TARGET_IMAGES['hearing_aid']},
    {'enabled': False, 'source_name': 'step_curb', 'target_class': 'step_curb', 'url_or_script': '', 'aliases': ['step_curb', 'step', 'curb', 'stairs'], 'target_total_images': TARGET_IMAGES['step_curb']},
    {'enabled': False, 'source_name': 'exit_sign', 'target_class': 'exit_sign', 'url_or_script': '', 'aliases': ['exit_sign', 'exit sign'], 'target_total_images': TARGET_IMAGES['exit_sign']},
    {'enabled': False, 'source_name': 'bathroom_sign', 'target_class': 'bathroom_sign', 'url_or_script': '', 'aliases': ['bathroom_sign', 'bathroom sign', 'restroom sign'], 'target_total_images': TARGET_IMAGES['bathroom_sign']},
    {'enabled': False, 'source_name': 'tupperware', 'target_class': 'tupperware', 'url_or_script': '', 'aliases': ['tupperware', 'food container', 'plastic container', 'storage container'], 'target_total_images': TARGET_IMAGES['tupperware']},
    {'enabled': False, 'source_name': 'jug', 'target_class': 'jug', 'url_or_script': '', 'aliases': ['jug'], 'target_total_images': TARGET_IMAGES['jug']},
    {'enabled': False, 'source_name': 'canned_good', 'target_class': 'canned_good', 'url_or_script': '', 'aliases': ['canned_good', 'canned good', 'can', 'tin can'], 'target_total_images': TARGET_IMAGES['canned_good']},
    {'enabled': False, 'source_name': 'fork', 'target_class': 'fork', 'url_or_script': '', 'aliases': ['fork'], 'target_total_images': TARGET_IMAGES['fork']},
    {'enabled': False, 'source_name': 'spoon', 'target_class': 'spoon', 'url_or_script': '', 'aliases': ['spoon'], 'target_total_images': TARGET_IMAGES['spoon']},
    {'enabled': False, 'source_name': 'knife', 'target_class': 'knife', 'url_or_script': '', 'aliases': ['knife'], 'target_total_images': TARGET_IMAGES['knife']},
    {'enabled': False, 'source_name': 'drinking_glass', 'target_class': 'drinking_glass', 'url_or_script': '', 'aliases': ['drinking_glass', 'drinking glass', 'glass'], 'target_total_images': TARGET_IMAGES['drinking_glass']},
    {'enabled': False, 'source_name': 'wine_glass', 'target_class': 'wine_glass', 'url_or_script': '', 'aliases': ['wine_glass', 'wine glass'], 'target_total_images': TARGET_IMAGES['wine_glass']},
    {'enabled': False, 'source_name': 'trash_can', 'target_class': 'trash_can', 'url_or_script': '', 'aliases': ['trash_can', 'trash can', 'waste bin', 'waste container'], 'target_total_images': TARGET_IMAGES['trash_can']},
    {'enabled': False, 'source_name': 'coffee_mug', 'target_class': 'coffee_mug', 'url_or_script': '', 'aliases': ['coffee_mug', 'coffee mug', 'mug'], 'target_total_images': TARGET_IMAGES['coffee_mug']},
    {'enabled': False, 'source_name': 'salt_pepper_shaker', 'target_class': 'salt_pepper_shaker', 'url_or_script': '', 'aliases': ['salt_pepper_shaker', 'salt and pepper shakers', 'shaker'], 'target_total_images': TARGET_IMAGES['salt_pepper_shaker']},
    {'enabled': False, 'source_name': 'measuring_cup', 'target_class': 'measuring_cup', 'url_or_script': '', 'aliases': ['measuring_cup', 'measuring cup'], 'target_total_images': TARGET_IMAGES['measuring_cup']},
    {'enabled': False, 'source_name': 'pan', 'target_class': 'pan', 'url_or_script': '', 'aliases': ['pan', 'frying pan'], 'target_total_images': TARGET_IMAGES['pan']},
    {'enabled': False, 'source_name': 'cellphone', 'target_class': 'cellphone', 'url_or_script': '', 'aliases': ['cellphone', 'cell phone', 'mobile phone', 'phone'], 'target_total_images': TARGET_IMAGES['cellphone']},
    {'enabled': False, 'source_name': 'purse', 'target_class': 'purse', 'url_or_script': '', 'aliases': ['purse', 'handbag'], 'target_total_images': TARGET_IMAGES['purse']},
    {'enabled': False, 'source_name': 'toilet', 'target_class': 'toilet', 'url_or_script': '', 'aliases': ['toilet'], 'target_total_images': TARGET_IMAGES['toilet']},
    {'enabled': False, 'source_name': 'toothbrush', 'target_class': 'toothbrush', 'url_or_script': '', 'aliases': ['toothbrush'], 'target_total_images': TARGET_IMAGES['toothbrush']},
    {'enabled': False, 'source_name': 'tv_remote', 'target_class': 'tv_remote', 'url_or_script': '', 'aliases': ['tv_remote', 'remote', 'remote control'], 'target_total_images': TARGET_IMAGES['tv_remote']},
    {'enabled': False, 'source_name': 'writing_device', 'target_class': 'writing_device', 'url_or_script': '', 'aliases': ['writing_device', 'pen', 'pencil'], 'target_total_images': TARGET_IMAGES['writing_device']},
    {'enabled': False, 'source_name': 'eyeglasses', 'target_class': 'eyeglasses', 'url_or_script': '', 'aliases': ['eyeglasses', 'glasses'], 'target_total_images': TARGET_IMAGES['eyeglasses']},
    {'enabled': False, 'source_name': 'thermostat', 'target_class': 'thermostat', 'url_or_script': '', 'aliases': ['thermostat'], 'target_total_images': TARGET_IMAGES['thermostat']},
    {'enabled': False, 'source_name': 'razor', 'target_class': 'razor', 'url_or_script': '', 'aliases': ['razor'], 'target_total_images': TARGET_IMAGES['razor']},
    {'enabled': False, 'source_name': 'window', 'target_class': 'window', 'url_or_script': '', 'aliases': ['window'], 'target_total_images': TARGET_IMAGES['window']},
    {'enabled': False, 'source_name': 'chair', 'target_class': 'chair', 'url_or_script': '', 'aliases': ['chair'], 'target_total_images': TARGET_IMAGES['chair']},
    {'enabled': False, 'source_name': 'table', 'target_class': 'table', 'url_or_script': '', 'aliases': ['table'], 'target_total_images': TARGET_IMAGES['table']},
    {'enabled': False, 'source_name': 'electrical_outlet', 'target_class': 'electrical_outlet', 'url_or_script': '', 'aliases': ['electrical_outlet', 'electrical outlet', 'power outlet', 'power plugs and sockets'], 'target_total_images': TARGET_IMAGES['electrical_outlet']},
    {'enabled': False, 'source_name': 'soap_dispenser', 'target_class': 'soap_dispenser', 'url_or_script': '', 'aliases': ['soap_dispenser', 'soap dispenser'], 'target_total_images': TARGET_IMAGES['soap_dispenser']},
    {'enabled': False, 'source_name': 'wallet', 'target_class': 'wallet', 'url_or_script': '', 'aliases': ['wallet'], 'target_total_images': TARGET_IMAGES['wallet']},
    {'enabled': False, 'source_name': 'toothpaste', 'target_class': 'toothpaste', 'url_or_script': '', 'aliases': ['toothpaste'], 'target_total_images': TARGET_IMAGES['toothpaste']},
    {'enabled': False, 'source_name': 'door', 'target_class': 'door', 'url_or_script': '', 'aliases': ['door', 'door_frame', 'door frame'], 'target_total_images': TARGET_IMAGES.get('door', 1500)},
    {'enabled': False, 'source_name': 'medicine_bottle', 'target_class': 'medicine_bottle', 'url_or_script': '', 'aliases': ['medicine_bottle', 'medicine bottle', 'medicine'], 'target_total_images': TARGET_IMAGES['medicine_bottle']},
    {'enabled': False, 'source_name': 'lemon', 'target_class': 'lemon', 'url_or_script': '', 'aliases': ['lemon'], 'target_total_images': TARGET_IMAGES['lemon']},
    {'enabled': False, 'source_name': 'deodorant', 'target_class': 'deodorant', 'url_or_script': '', 'aliases': ['deodorant'], 'target_total_images': TARGET_IMAGES['deodorant']},
    {'enabled': False, 'source_name': 'keys', 'target_class': 'keys', 'url_or_script': '', 'aliases': ['keys', 'key'], 'target_total_images': TARGET_IMAGES['keys']},
]

enabled_sources = [
    s for s in ROBOFLOW_SOURCES
    if str(s.get('url_or_script', '')).strip() and not s.get('skip', False)
]
for s in enabled_sources:
    assert s['target_class'] in name_to_id, f"Unknown target_class for this checkpoint: {s['target_class']}"
print('Enabled Roboflow sources:', len(enabled_sources))
for s in enabled_sources:
    print('-', s['source_name'], '->', s['target_class'])


In [ ]:
# Cell 6: Download and inspect enabled Roboflow sources

def norm_name(x):
    return re.sub(r'[^a-z0-9]+', ' ', str(x).lower()).strip()

def resolve_source_mapping(src, source_names):
    explicit_ids = src.get('source_class_ids')
    if explicit_ids is not None:
        mapped_ids = sorted({int(i) for i in explicit_ids})
        invalid = [i for i in mapped_ids if i < 0 or i >= len(source_names)]
        if invalid:
            raise ValueError(
                f"{src['source_name']} has invalid source_class_ids {invalid}; "
                f"available ids are 0..{len(source_names) - 1}"
            )
        return mapped_ids, False, 'explicit_source_class_ids'

    alias_norms = {norm_name(a) for a in src.get('aliases', [])}
    mapped_ids = [i for i, n in enumerate(source_names) if norm_name(n) in alias_norms]
    if mapped_ids:
        return mapped_ids, False, 'alias_match'

    if len(source_names) == 1:
        return [0], True, 'single_class_fallback'

    return [], False, 'unresolved_multiclass_mapping'

def extract_roboflow_url(url_or_script):
    text = str(url_or_script).strip()
    m = re.search(r'https://(?:app|universe)\.roboflow\.com/ds/[^\s"\']+', text)
    if m:
        return m.group(0)
    if text.startswith('http'):
        return text
    raise ValueError(f'Could not find Roboflow download URL in: {text[:160]}')

def download_roboflow_source(src, overwrite=False):
    out_dir = ROBOSRC_ROOT / src['source_name']
    zip_path = ROBOSRC_ROOT / f"{src['source_name']}.zip"
    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    if out_dir.exists() and list(out_dir.rglob('data.yaml')):
        print('Already downloaded:', src['source_name'])
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    url = extract_roboflow_url(src['url_or_script'])
    print('Downloading', src['source_name'])
    result = subprocess.run(
        [
            'curl', '--fail', '--location', '--show-error',
            '--retry', '3', '--retry-all-errors',
            '--output', str(zip_path), url,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        zip_path.unlink(missing_ok=True)
        detail = (result.stderr or result.stdout).strip()
        if '403' in detail or '401' in detail:
            raise RuntimeError(
                f"Roboflow rejected the export link for {src['source_name']}. "
                'Generate a fresh YOLO26 download command in Roboflow, replace '
                "url_or_script, and rerun this cell. Signed export links can expire. "
                f"curl output: {detail}"
            )
        raise RuntimeError(
            f"Roboflow download failed for {src['source_name']} "
            f"(curl exit {result.returncode}): {detail}"
        )
    if not zip_path.exists() or zip_path.stat().st_size == 0:
        raise RuntimeError(f"Roboflow returned an empty download for {src['source_name']}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    zip_path.unlink(missing_ok=True)
    return out_dir

def find_data_yaml(source_dir):
    candidates = list(Path(source_dir).rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found in {source_dir}')
    return sorted(candidates, key=lambda p: len(p.parts))[0]

for src in enabled_sources:
    download_roboflow_source(src, overwrite=False)

inspect_rows = []
for src in enabled_sources:
    data_yaml_path = find_data_yaml(ROBOSRC_ROOT / src['source_name'])
    data, names = load_yolo_names(data_yaml_path)
    mapped_ids, single_class_fallback, mapping_status = resolve_source_mapping(src, names)
    split_counts = {}
    label_counts = Counter()
    for split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / split / 'images'
        lbl_dir = data_yaml_path.parent / split / 'labels'
        imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS] if img_dir.exists() else []
        split_counts[split] = len(imgs)
        if lbl_dir.exists():
            for lbl in lbl_dir.glob('*.txt'):
                for line in lbl.read_text().splitlines():
                    if line.strip():
                        cls_id = int(float(line.split()[0]))
                        label_counts[names[cls_id] if 0 <= cls_id < len(names) else f'BAD_ID_{cls_id}'] += 1
    inspect_rows.append({
        'source_name': src['source_name'],
        'target_class': src['target_class'],
        'data_yaml': str(data_yaml_path),
        'source_classes': names,
        'mapped_source_class_ids': mapped_ids,
        'mapped_source_class_names': [names[i] for i in mapped_ids],
        'mapping_status': mapping_status,
        'single_class_fallback_would_be_used': single_class_fallback,
        'train_images': split_counts.get('train', 0),
        'valid_images': split_counts.get('valid', 0),
        'test_images': split_counts.get('test', 0),
        'source_label_counts': dict(label_counts),
    })

inspect_df = pd.DataFrame(inspect_rows)
inspect_df.to_csv(WORK_META / 'roboflow_source_inspection.csv', index=False)
display(inspect_df)

unresolved = inspect_df[inspect_df['mapping_status'] == 'unresolved_multiclass_mapping']
if len(unresolved):
    print('\nACTION REQUIRED: these multi-class datasets could not be mapped safely.')
    print('Add the exact source class name to aliases, or set source_class_ids after reviewing source_classes.')
    display(unresolved[['source_name', 'target_class', 'source_classes', 'mapped_source_class_ids', 'mapping_status']])


In [ ]:
# Cell 7: Quality filters and append import

SMALL_CLASSES = {
    'keys', 'medicine_bottle', 'electrical_outlet', 'light_switch',
    'toothpaste', 'deodorant', 'razor', 'hearing_aid', 'wallet',
    'thermostat', 'salt_pepper_shaker', 'measuring_cup',
}
THIN_CLASSES = {'cane', 'railing', 'keys', 'writing_device', 'step_curb'}

def file_sha1(path, chunk_size=1024 * 1024):
    h = hashlib.sha1()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def existing_image_hashes():
    hashes = set()
    for split in ['train', 'valid', 'test']:
        img_dir = WORK_DS / split / 'images'
        if not img_dir.exists():
            continue
        for img_path in img_dir.glob('*'):
            if img_path.suffix.lower() in IMAGE_EXTS:
                try:
                    hashes.add(file_sha1(img_path))
                except Exception:
                    pass
    return hashes

def current_class_image_counts():
    counts = Counter()
    for split in ['train', 'valid', 'test']:
        lbl_dir = WORK_DS / split / 'labels'
        if not lbl_dir.exists():
            continue
        for lbl_path in lbl_dir.glob('*.txt'):
            seen = set()
            for line in lbl_path.read_text().splitlines():
                if line.strip():
                    cls_id = int(float(line.split()[0]))
                    if 0 <= cls_id < len(NAMES):
                        seen.add(NAMES[cls_id])
            for cls in seen:
                counts[cls] += 1
    return counts

def split_for_source(source_name, file_name):
    h = int(hashlib.md5(f'trial8-append-{source_name}-{file_name}'.encode()).hexdigest(), 16) % 100
    if h < 80:
        return 'train'
    if h < 90:
        return 'valid'
    return 'test'

def image_quality_ok(img_path):
    try:
        im = ImageOps.exif_transpose(Image.open(img_path)).convert('L')
        arr = np.asarray(im.resize((256, 256)))
        mean = float(arr.mean())
        std = float(arr.std())
        sharpness = float(
            np.abs(np.diff(arr.astype(np.float32), axis=1)).mean()
            + np.abs(np.diff(arr.astype(np.float32), axis=0)).mean()
        )
        reasons = []
        if mean < 28 or mean > 228:
            reasons.append('bad_brightness')
        if std < 16:
            reasons.append('low_contrast')
        if sharpness < 5.0:
            reasons.append('likely_blurry')
        if img_path.stat().st_size < 7000:
            reasons.append('tiny_file')
        return len(reasons) == 0, {'brightness': mean, 'contrast': std, 'sharpness': sharpness, 'reject_reason': ','.join(reasons)}
    except Exception as e:
        return False, {'brightness': None, 'contrast': None, 'sharpness': None, 'reject_reason': f'decode_error:{e}'}

def parse_source_labels(lbl_path, source_names, mapped_ids, target_class, allow_single_class_fallback):
    raw_lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
    if not raw_lines:
        return [], ['empty_label']
    out = []
    reasons = []
    for line in raw_lines:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            reasons.append('bad_column_count')
            continue
        try:
            src_cls = int(float(parts[0]))
            x, y, w, h = [float(v) for v in parts[1:]]
        except Exception:
            reasons.append('parse_error')
            continue
        if src_cls < 0 or src_cls >= len(source_names):
            reasons.append('bad_class_id')
            continue
        is_target = src_cls in mapped_ids or (allow_single_class_fallback and len(source_names) == 1)
        if not is_target:
            reasons.append(f'non_target_source_class:{source_names[src_cls]}')
            continue
        if not all(math.isfinite(v) for v in [x, y, w, h]):
            reasons.append('nonfinite_coord')
            continue
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            reasons.append('coord_out_of_range')
            continue
        area = w * h
        aspect = max(w / max(h, 1e-6), h / max(w, 1e-6))
        min_area = 0.00012 if target_class in SMALL_CLASSES else 0.0005
        max_aspect = 28 if target_class in THIN_CLASSES else 16
        if area < min_area:
            reasons.append('box_too_small')
            continue
        if area > 0.90:
            reasons.append('box_too_large')
            continue
        if aspect > max_aspect:
            reasons.append('box_aspect_extreme')
            continue
        out.append(f'{name_to_id[target_class]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    if not out and not reasons:
        reasons.append('no_usable_target_label')
    return out, reasons

current_counts = current_class_image_counts()
known_hashes = existing_image_hashes()
selected_rows, rejected_rows, summary_rows = [], [], []

for src in enabled_sources:
    source_name = src['source_name']
    target_class = src['target_class']
    needed = max(0, int(src['target_total_images']) - current_counts.get(target_class, 0))
    if needed <= 0 and not src.get('allow_above_target', False):
        print(f'Skipping {source_name}: {target_class} already at or above target.')
        continue

    data_yaml_path = find_data_yaml(ROBOSRC_ROOT / source_name)
    data, source_names = load_yolo_names(data_yaml_path)
    mapped_ids, allow_single, mapping_status = resolve_source_mapping(src, source_names)
    if not mapped_ids:
        print(
            f'Skipping {source_name}: unresolved class mapping. '
            'Review Cell 6 and update aliases or source_class_ids.'
        )
        summary_rows.append({
            'source_name': source_name,
            'target_class': target_class,
            'needed_before_import': needed,
            'accepted_candidates': 0,
            'selected': 0,
            'target_total_images': src['target_total_images'],
            'total_after_import_estimate': current_counts.get(target_class, 0),
            'source_classes': source_names,
            'mapping_status': mapping_status,
            'single_class_fallback_used': allow_single,
        })
        continue

    candidates = []
    for original_split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / original_split / 'images'
        lbl_dir = data_yaml_path.parent / original_split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            try:
                sha1 = file_sha1(img_path)
            except Exception:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': 'hash_read_error'})
                continue
            if sha1 in known_hashes:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': 'duplicate_image_sha1'})
                continue
            lbl_path = lbl_dir / f'{img_path.stem}.txt'
            lines_out, label_reasons = parse_source_labels(lbl_path, source_names, mapped_ids, target_class, allow_single)
            has_non_target_labels = any(r.startswith('non_target_source_class:') for r in label_reasons)
            if has_non_target_labels and src.get('reject_mixed_class_images', True):
                rejected_rows.append({
                    'source_name': source_name,
                    'target_class': target_class,
                    'file_name': img_path.name,
                    'reason': 'mixed_class_image_rejected:' + ';'.join(sorted(set(label_reasons))),
                })
                continue
            if not lines_out:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': ';'.join(sorted(set(label_reasons)))})
                continue
            ok, q = image_quality_ok(img_path)
            if not ok:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': q['reject_reason']})
                continue
            label_count = len(lines_out)
            if label_count > 20:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': f'too_many_labels:{label_count}'})
                continue
            score = min(q['contrast'], 90) * 0.25 + min(q['sharpness'], 45) * 0.75 + (8 if 1 <= label_count <= 5 else -4)
            candidates.append({'img_path': img_path, 'sha1': sha1, 'source_name': source_name, 'target_class': target_class, 'original_split': original_split, 'label_lines': lines_out, 'label_count': label_count, 'score': score, **q})

    max_images = int(src.get('max_images_from_source') or needed)
    pick_n = max_images if src.get('allow_above_target', False) else min(needed, max_images)
    picked = sorted(candidates, key=lambda r: r['score'], reverse=True)[:pick_n]

    selected_before = len(selected_rows)
    for row in tqdm(picked, desc=f'Appending {source_name}->{target_class}'):
        img_path = row['img_path']
        out_split = split_for_source(source_name, img_path.name)
        out_stem = f"rf8a_{source_name}_{row['sha1'][:10]}_{img_path.stem}"
        out_img = WORK_DS / out_split / 'images' / f'{out_stem}.jpg'
        out_lbl = WORK_DS / out_split / 'labels' / f'{out_stem}.txt'
        if out_img.exists() or out_lbl.exists():
            rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': 'output_name_exists'})
            continue
        im = ImageOps.exif_transpose(Image.open(img_path)).convert('RGB')
        im = im.resize((640, 640), Image.Resampling.LANCZOS)
        im.save(out_img, quality=92)
        out_lbl.write_text('\n'.join(row['label_lines']) + '\n')
        known_hashes.add(row['sha1'])
        selected_rows.append({'source_name': source_name, 'target_class': target_class, 'split': out_split, 'original_split': row['original_split'], 'file_name': out_img.name, 'label_count': row['label_count'], 'score': row['score'], 'brightness': row['brightness'], 'contrast': row['contrast'], 'sharpness': row['sharpness']})

    imported_count = len(selected_rows) - selected_before
    current_counts[target_class] += imported_count
    summary_rows.append({'source_name': source_name, 'target_class': target_class, 'needed_before_import': needed, 'accepted_candidates': len(candidates), 'selected': imported_count, 'target_total_images': src['target_total_images'], 'total_after_import_estimate': current_counts[target_class], 'source_classes': source_names, 'mapping_status': mapping_status, 'single_class_fallback_used': allow_single})

selected_df = pd.DataFrame(selected_rows)
rejected_df = pd.DataFrame(rejected_rows)
source_summary_df = pd.DataFrame(summary_rows)
selected_df.to_csv(WORK_META / 'roboflow_append_selected_import_manifest.csv', index=False)
rejected_df.to_csv(WORK_META / 'roboflow_append_rejected_import_manifest.csv', index=False)
source_summary_df.to_csv(WORK_META / 'roboflow_append_import_summary.csv', index=False)

print('Selected images:', len(selected_df))
display(source_summary_df)
print('Top rejection reasons:')
display(rejected_df['reason'].value_counts().head(30) if len(rejected_df) else pd.Series(dtype=int))


In [ ]:
# Cell 8: Re-audit after import and report what is still needed

split_summary, class_summary, bad_labels = audit_dataset()
split_summary.to_csv(WORK_META / 'latest_split_summary.csv', index=False)
class_summary.to_csv(WORK_META / 'latest_class_summary.csv', index=False)
bad_labels.to_csv(WORK_META / 'latest_bad_labels.csv', index=False)

missing = class_summary[class_summary['total_images'] == 0].copy()
needs_more = class_summary[class_summary['still_need'] > 0].copy()
needed_report = {
    'created_at': STAMP,
    'source_checkpoint': str(SOURCE_ZIP),
    'class_count': len(NAMES),
    'total_images': int(split_summary['images'].sum()),
    'total_boxes': int(class_summary['total_boxes'].sum()),
    'bad_label_rows': int(len(bad_labels)),
    'missing_classes': missing['class'].tolist(),
    'classes_below_goal': needs_more[['class', 'total_images', 'goal_images', 'still_need']].to_dict(orient='records'),
}
(WORK_META / 'latest_needed_report.json').write_text(json.dumps(needed_report, indent=2))

display(split_summary)
display(needs_more[['class', 'total_images', 'goal_images', 'still_need', 'total_boxes']])
print('Missing classes:', missing['class'].tolist())
print('Bad label rows:', len(bad_labels))
print('Saved:', WORK_META / 'latest_needed_report.json')


In [ ]:
# Cell 9: Count chart

sns.set_theme(style='whitegrid')
chart_dir = WORK_META / 'charts'
chart_dir.mkdir(exist_ok=True)

plot_df = class_summary.sort_values('total_images', ascending=True).copy()
plt.figure(figsize=(11, max(8, len(plot_df) * 0.22)))
sns.barplot(data=plot_df, y='class', x='total_images', color='steelblue')
for pos, (_, r) in enumerate(plot_df.iterrows()):
    plt.plot([r['goal_images'], r['goal_images']], [pos - 0.35, pos + 0.35], color='firebrick', linewidth=2)
plt.title('Trial 8 Roboflow Append: Images per Class')
plt.xlabel('images containing class; red tick = goal')
plt.ylabel('')
plt.tight_layout()
plt.savefig(chart_dir / 'class_counts_after_append_vs_goal.png', dpi=180)
plt.show()


In [ ]:
# Cell 10: Zip updated dataset checkpoint to Drive

ZIP_LOCAL = Path(f'/content/galvat_trial8_roboflow_appended_{STAMP}.zip')
ZIP_DRIVE = DRIVE_OUT / ZIP_LOCAL.name

checkpoint_summary = {
    'created_at': STAMP,
    'source_checkpoint': str(SOURCE_ZIP),
    'work_dataset': str(WORK_DS),
    'metadata_root': str(WORK_META),
    'dataset_policy': 'Started from Roboflow-only Trial 8 checkpoint and appended enabled Roboflow sources without overwriting existing images.',
    'class_count': len(NAMES),
    'total_images': int(split_summary['images'].sum()),
    'total_boxes': int(class_summary['total_boxes'].sum()),
    'bad_label_rows': int(len(bad_labels)),
    'primary_metric_for_future_training': 'GALVAT weighted score',
}
(WORK_META / 'checkpoint_summary.json').write_text(json.dumps(checkpoint_summary, indent=2))

if ZIP_LOCAL.exists():
    ZIP_LOCAL.unlink()

INTERNAL_ROOT = Path('galvat_trial8_roboflow_appended')
with zipfile.ZipFile(ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in tqdm(list(WORK_ROOT.rglob('*')), desc='Zipping appended checkpoint'):
        if path.is_file():
            z.write(path, INTERNAL_ROOT / path.relative_to(WORK_ROOT))

print('Local zip:', ZIP_LOCAL)
print('Zip size GB:', ZIP_LOCAL.stat().st_size / 1e9)
print('Copying to Drive:', ZIP_DRIVE)
shutil.copy2(ZIP_LOCAL, ZIP_DRIVE)
print('Saved:', ZIP_DRIVE)
print(json.dumps(checkpoint_summary, indent=2))
